# 02 · Feature Engineering — PySpark Pipeline

Runs the distributed PySpark feature-engineering pipeline on the raw EIA 930 CSVs.

**Computes per BA per hour:**
- Lag features: `solar_lag_1h` … `solar_lag_48h`, same for wind and demand (144 cols)
- Rolling statistics: 6h / 24h / 7d mean + std for solar, wind, demand (18 cols)
- Mismatch label (4-class severity)
- Lead targets: solar and wind generation at t+6h, t+12h, t+24h

**Outputs**: `data/processed/features.parquet` (partitioned by year and BA)

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
from src.features import RAW_DIR, PROCESSED_DIR, _get_spark

csv_files = sorted(RAW_DIR.glob('EIA930_BALANCE_*.csv'))
print(f'Raw CSV files: {len(csv_files)}')
for f in csv_files:
    print(f'  {f.name}')

In [ ]:
# Verify Spark session configuration
spark = _get_spark()
print('Spark version:', spark.version)
print('Master:', spark.sparkContext.master)
print('App name:', spark.sparkContext.appName)
conf = spark.sparkContext.getConf()
print('driver.memory:', conf.get('spark.driver.memory'))
print('shuffle.partitions:', conf.get('spark.sql.shuffle.partitions'))

In [ ]:
# Quick schema inspection before full pipeline run
if csv_files:
    raw = spark.read.option('header', 'true').csv(str(csv_files[0]))
    print(f'Raw columns ({len(raw.columns)}):',  raw.columns)
    raw.show(3, truncate=False)

In [ ]:
# Run the full feature pipeline
# NOTE: On 50-80M rows this takes ~20-60 minutes depending on hardware.
# Pass a fraction (e.g. 0.1) to run on 10% of data for a quick test.
from src.features import run_features

run_features(sample_frac=1.0)

In [ ]:
# Inspect the output parquet
import pyarrow.dataset as ds
import pandas as pd

dataset = ds.dataset(
    str(PROCESSED_DIR / 'features.parquet'),
    format='parquet', partitioning='hive'
)
print(f'Parquet partitions: {len(dataset.files)}')
print(f'Total columns: {len(dataset.schema.names)}')
print()
print('Schema:')
print(dataset.schema)

In [ ]:
# Row count per year
sample = dataset.to_table(columns=['year', 'ba']).to_pandas()
print('Rows per year:')
print(sample.groupby('year').size())
print(f'\nTotal rows: {len(sample):,}')
print(f'Unique BAs: {sample["ba"].nunique()}')

In [ ]:
# Mismatch class distribution
labels = dataset.to_table(columns=['mismatch_label']).to_pandas()
vc = labels['mismatch_label'].value_counts().sort_index()
label_names = {0: 'Balanced', 1: 'Mod surplus', 2: 'Sev surplus', 3: 'Deficit'}
for k, v in vc.items():
    print(f'  Class {int(k)} ({label_names.get(int(k), "?"):12s}): {v:>10,}  ({v/len(labels)*100:.1f}%)')